# **03MIAR: Proyecto Algoritmos de optimización (Grupo 19)**<br>
**Nombre y Apellidos (Miembro 1):** *Michael Cristian Pospesch Isfan*<br>
**Nombre y Apellidos (Miembro 2):** *Daniel Enrique Guzman Reyes*<br>
GoogleColab: https://colab.research.google.com/drive/1CmqMmxXeE4uHGhXziPi6Ojn9c7vNvbwN?usp=sharing<br>
GitHub: https://github.com/MikiMCP/Algoritmos-de-optimizacion/blob/main/03MIAR_PROYECTO_GRUPO19.ipynb<br>
**Problema:**
### **1. Sesiones de doblaje** <br>

**Descripción del problema:** <br>
Se precisa coordinar el doblaje de una película. Los actores del doblaje deben coincidir en las tomas en las que sus personajes aparecen juntos en las diferentes tomas. Los actores de doblaje cobran todos la misma cantidad por cada día que deben desplazarse hasta el estudio de grabación independientemente del número de tomas que se graben. No es posible grabar más de 6 tomas por día. El objetivo es planificar las sesiones por día de manera que el gasto por los servicios de los actores de doblaje sea el menor  osible.                                        

#### **¿Cuántas posibilidades hay sin tener en cuenta las restricciones?**<br>

Si eliminamos la restricción de grabar un máximo de 6 tomas por día, nos encontramos con un problema de combinatoria muy grande. En el extremo donde disponemos de hasta 30 días distintos de grabación, cada una de las 30 tomas podría ser asignada a cualquiera de esos 30 días.
Matemáticamente, esto se modela como variaciones con repetición, donde la base son los 30 días y el exponente son las 30 tomas. La cantidad de calendarios posibles sería $30^{30}$, generando un espacio de soluciones de orden exponencial que demuestra la inviabilidad de usar el método de fuerza bruta.

In [ ]:
# Cálculo de posibilidades sin restricciones
tomas = 30
dias_posibles = 30
posibilidades_sin_restricciones = dias_posibles ** tomas

print(f"Posibilidades sin restricciones: {posibilidades_sin_restricciones}")

Posibilidades sin restricciones: 205891132094649000000000000000000000000000000


#### **¿Cuántas posibilidades hay teniendo en cuenta todas las restricciones?<br>**

Al aplicar la restricción de un máximo de 6 tomas por día, logramos acotar el problema; sin embargo, el espacio de búsqueda resultante sigue siendo computacionalmente intratable.El Escenario de Máxima EficienciaSi buscamos la opción más ajustada, es decir, repartir las 30 tomas en el mínimo estricto de 5 días (a razón de 6 tomas diarias), el cálculo del número de combinaciones posibles se define mediante el producto de combinaciones sin repetición:Día 1: Elegimos 6 tomas de las 30 disponibles.Día 2: Elegimos 6 de las 24 restantes.Días sucesivos: Continuamos el proceso hasta agotar las tomas.Formulación MatemáticaLa cantidad total de combinaciones posibles ($N$) se expresa mediante la siguiente serie de coeficientes binomiales:

$$N = \binom{30}{6} \times \binom{24}{6} \times \binom{18}{6} \times \binom{12}{6} \times \binom{6}{6}$$

O, de forma simplificada mediante el coeficiente multinomial:

$$N = \frac{30!}{6! \cdot 6! \cdot 6! \cdot 6! \cdot 6!} \approx 1.37 \times 10^{17}$$

El resultado supera los 137 trillones de combinaciones. Esta cifra confirma que el uso de Fuerza Bruta es un enfoque totalmente ineficiente para resolver este problema, lo que justifica la necesidad de utilizar algoritmos de optimización o heurísticas.


In [ ]:
import math
# Cálculo de combinaciones para 5 días de 6 tomas cada uno
c1 = math.comb(30, 6)
c2 = math.comb(24, 6)
c3 = math.comb(18, 6)
c4 = math.comb(12, 6)
c5 = math.comb(6, 6)

posibilidades_con_restricciones = c1 * c2 * c3 * c4 * c5
print(f"Posibilidades con restricciones (Ej. 5 días): {posibilidades_con_restricciones}")

Posibilidades con restricciones (Ej. 5 días): 1370874167589326400


#### **Modelo para el espacio de soluciones**<br>
(*) ¿Cual es la estructura de datos que mejor se adapta al problema? Argumentalo.(Es posible que hayas elegido una al principio y veas la necesidad de cambiar, arguentalo)


1. Matriz binaria de participación (lista de listas): el tamaño de la matriz (R) será de T(tomas) x A(actores) = 30 x 10. Por ende $R[t][a] = 1$ (si el actor "a" participa en la toma "t") o 0 (si no).<br>
Argumento: necesito una estructura donde tener todas las tomas que se graban y que actores participan en ellas.<br>
2. Matriz binaria de decisión: Como quiero que el resultado sea "Día d: tomas..." y el coste total, necesito dos familias de variables:
-  Matriz (X) toma-dia: $x[t][d] = 1$ (si la toma "t" se graba el día "d") o 0 (si no)
- Matriz (Y) actor-dia: $y[a][d] = 1$ (si el actor "a" asiste el día "d") o 0 (si no)<br>
Fijaré los días como D = 30 (el máximo), los días no usados quedarán a 0. Entonces el tamaño de estas matrices es: $X[30][D]$ y $𝑌[10][𝐷]$

Y la salida se mostrará como:<br>
- por día: lista tomas con $X[t][d]=1$
- coste: $∑Y[a][d]$


#### **2. Según el modelo para el espacio de soluciones<br>**
(*)¿Cual es la función objetivo?

(*)¿Es un problema de maximización o minimización?

Como cada actor cobra una vez por cada día que se desplaza, el coste total es el número total de pares $(\text{actor}, \text{día})$ en los que ese actor asiste al rodaje.

1. Definimos la variable binaria.
$$y_{a,d} \in \{0,1\}$$

    Donde: $y_{a,d} = 1$ si el actor $a$ está presente en el día de rodaje $d$.$y_{a,d} = 0$ en caso contrario.

2. El objetivo es minimizar el sumatorio de todas las asistencias.

$$\min Z = \sum_{a \in A} \sum_{d \in D} y_{a,d}$$

Nota: Esto equivale a "minimizar el total de desplazamientos pagados". Por lo tanto, se trata de un problema de minimización, ya que buscamos reducir al máximo los costes operativos derivados de los desplazamientos de los actores.

(*)Diseña un algoritmo que mejore la complejidad del algortimo por fuerza bruta. Argumenta porque crees que mejora el algoritmo por fuerza bruta

In [ ]:
# =========================
# SOLUCION DEL PROBLEMA
# =========================
### Librerias importadas
import numpy as np
from scipy.optimize import milp, LinearConstraint, Bounds
from scipy.sparse import lil_matrix
import itertools

In [ ]:
# =========================
# 1) DEFINICIÓN DE MATRICES
# =========================
R = [
  [1, 1, 1, 1, 1, 0, 0, 0, 0, 0],
  [0, 0, 1, 1, 1, 0, 0, 0, 0, 0],
  [0, 1, 0, 0, 1, 0, 1, 0, 0, 0],
  [1, 1, 0, 0, 0, 0, 1, 1, 0, 0],
  [0, 1, 0, 1, 0, 0, 0, 1, 0, 0],
  [1, 1, 0, 1, 1, 0, 0, 0, 0, 0],
  [1, 1, 0, 1, 1, 0, 0, 0, 0, 0],
  [1, 1, 0, 0, 0, 1, 0, 0, 0, 0],
  [1, 1, 0, 1, 0, 0, 0, 0, 0, 0],
  [1, 1, 0, 0, 0, 1, 0, 0, 1, 0],
  [1, 1, 1, 0, 1, 0, 0, 1, 0, 0],
  [1, 1, 1, 1, 0, 1, 0, 0, 0, 0],
  [1, 0, 0, 1, 1, 0, 0, 0, 0, 0],
  [1, 0, 1, 0, 0, 1, 0, 0, 0, 0],
  [1, 1, 0, 0, 0, 0, 1, 0, 0, 0],
  [0, 0, 0, 1, 0, 0, 0, 0, 0, 1],
  [1, 0, 1, 0, 0, 0, 0, 0, 0, 0],
  [0, 0, 1, 0, 0, 1, 0, 0, 0, 0],
  [1, 0, 1, 0, 0, 0, 0, 0, 0, 0],
  [1, 0, 1, 1, 1, 0, 0, 0, 0, 0],
  [0, 0, 0, 0, 0, 1, 0, 1, 0, 0],
  [1, 1, 1, 1, 0, 0, 0, 0, 0, 0],
  [1, 0, 1, 0, 0, 0, 0, 0, 0, 0],
  [0, 0, 1, 0, 0, 1, 0, 0, 0, 0],
  [1, 1, 0, 1, 0, 0, 0, 0, 0, 1],
  [1, 0, 1, 0, 1, 0, 0, 0, 1, 0],
  [0, 0, 0, 1, 1, 0, 0, 0, 0, 0],
  [1, 0, 0, 1, 0, 0, 0, 0, 0, 0],
  [1, 0, 0, 0, 1, 1, 0, 0, 0, 0],
  [1, 0, 0, 1, 0, 0, 0, 0, 0, 0]
]
R_np = np.array(R, dtype=int)
T, A = 30, 10
D = 30

In [ ]:
# =========================
# 2) ÍNDICES DE VARIABLES
# =========================
def idx_x(t, d): return t * D + d
def idx_y(a, d): return T * D + a * D + d

n_x = T * D
n_y = A * D
n_u = D
n_vars = n_x + n_y + n_u

def idx_u(d): return n_x + n_y + d

In [ ]:
# =========================
# 3) MODELO LINEAL ENTERO
# =========================
c = np.zeros(n_vars, dtype=float)
for a in range(A):
    for d in range(D):
        c[idx_y(a, d)] = 1.0

integrality = np.ones(n_vars, dtype=int)
bounds = Bounds(lb=np.zeros(n_vars), ub=np.ones(n_vars))

constraints = []

# (1) Cada toma exactamente un día
A_eq = lil_matrix((T, n_vars), dtype=float)
b_eq = np.ones(T, dtype=float)
for t in range(T):
    for d in range(D):
        A_eq[t, idx_x(t, d)] = 1.0
constraints.append(LinearConstraint(A_eq.tocsr(), lb=b_eq, ub=b_eq))

# (2) Capacidad: sum x[t,d] <= 6*u[d]
A_cap = lil_matrix((D, n_vars), dtype=float)
lb_cap = np.full(D, -np.inf, dtype=float)
ub_cap = np.zeros(D, dtype=float)
for d in range(D):
    for t in range(T):
        A_cap[d, idx_x(t, d)] = 1.0
    A_cap[d, idx_u(d)] = -6.0
constraints.append(LinearConstraint(A_cap.tocsr(), lb=lb_cap, ub=ub_cap))

# (3) Link x<=y si el actor participa
link_rows = []
for t in range(T):
    for a in range(A):
        if R[t][a] == 1:
            for d in range(D):
                link_rows.append((t, a, d))

m_link = len(link_rows)
A_link = lil_matrix((m_link, n_vars), dtype=float)
lb_link = np.full(m_link, -np.inf, dtype=float)
ub_link = np.zeros(m_link, dtype=float)
for i, (t, a, d) in enumerate(link_rows):
    A_link[i, idx_x(t, d)] = 1.0
    A_link[i, idx_y(a, d)] = -1.0
constraints.append(LinearConstraint(A_link.tocsr(), lb=lb_link, ub=ub_link))

# (4) u[d] ordenado
m_ord = D - 1
A_ord = lil_matrix((m_ord, n_vars), dtype=float)
lb_ord = np.full(m_ord, -np.inf, dtype=float)
ub_ord = np.zeros(m_ord, dtype=float)
for d in range(D - 1):
    A_ord[d, idx_u(d)] = -1.0
    A_ord[d, idx_u(d + 1)] = 1.0
constraints.append(LinearConstraint(A_ord.tocsr(), lb=lb_ord, ub=ub_ord))

# (5) u[0]=1
A_u0 = lil_matrix((1, n_vars), dtype=float)
A_u0[0, idx_u(0)] = 1.0
constraints.append(LinearConstraint(A_u0.tocsr(), lb=np.array([1.0]), ub=np.array([1.0])))

# (6) No días usados vacíos: sum x[t,d] >= u[d]  -> u - sum x <= 0
A_used = lil_matrix((D, n_vars), dtype=float)
lb_used = np.full(D, -np.inf, dtype=float)
ub_used = np.zeros(D, dtype=float)
for d in range(D):
    A_used[d, idx_u(d)] = 1.0
    for t in range(T):
        A_used[d, idx_x(t, d)] = -1.0
constraints.append(LinearConstraint(A_used.tocsr(), lb=lb_used, ub=ub_used))

# (7) Tighten: y[a,d] <= sum_t R[t,a]*x[t,d]
m_tight = A * D
A_tight = lil_matrix((m_tight, n_vars), dtype=float)
lb_tight = np.full(m_tight, -np.inf, dtype=float)
ub_tight = np.zeros(m_tight, dtype=float)
row = 0
for a in range(A):
    for d in range(D):
        A_tight[row, idx_y(a, d)] = 1.0
        for t in range(T):
            if R[t][a] == 1:
                A_tight[row, idx_x(t, d)] = -1.0
        row += 1
constraints.append(LinearConstraint(A_tight.tocsr(), lb=lb_tight, ub=ub_tight))

# (8) Orden por carga: sum x[t,d] >= sum x[t,d+1]
m_load = D - 1
A_load = lil_matrix((m_load, n_vars), dtype=float)
lb_load = np.full(m_load, -np.inf, dtype=float)
ub_load = np.zeros(m_load, dtype=float)
for d in range(D - 1):
    for t in range(T):
        A_load[d, idx_x(t, d)] = -1.0
        A_load[d, idx_x(t, d + 1)] = 1.0
constraints.append(LinearConstraint(A_load.tocsr(), lb=lb_load, ub=ub_load))

In [ ]:
# =========================
# 4) RESOLVER (permitiendo solución no óptima)
# =========================
res = milp(
    c=c,
    integrality=integrality,
    bounds=bounds,
    constraints=constraints,
    options={
        "time_limit": 600,       # ajusta el tiempo máximo de ejecución.
        "mip_rel_gap": 0.05     # 5% gap => más fácil que devuelva algo viable rápido
    }
)

# ACEPTAMOS solución si existe res.x aunque no sea óptima
if res.x is None:
    raise RuntimeError(f"No se encontró ninguna solución factible. Estado: {res.status} | Msg: {res.message}")

z = res.x
cost = float(z @ c)

print(f"Estado solver: {res.status} | Msg: {res.message}")
print("NOTA: si pone 'Time limit reached', esta solución puede NO ser óptima, pero es factible.\n")

Estado solver: 1 | Msg: Time limit reached. (HiGHS Status 13: Time limit reached)
NOTA: si pone 'Time limit reached', esta solución puede NO ser óptima, pero es factible.



In [ ]:
# =========================
# 5) CONSTRUCCIÓN DE SALIDA
# =========================
schedule = []
for d in range(D):
    if z[idx_u(d)] <= 0.5:
        continue
    shots_d = [t + 1 for t in range(T) if z[idx_x(t, d)] > 0.5]
    if shots_d:
        schedule.append((d + 1, shots_d))

print("PLAN (por días):")
for day, shots in schedule:
    print(f"  Día {day}: tomas {', '.join(map(str, shots))}")

print("\nCOSTE TOTAL (actor-días pagados):", int(round(cost)))

print("\nDetalle (actores que vienen por día):")
for day, shots in schedule:
    d = day - 1
    actors_present = [a + 1 for a in range(A) if z[idx_y(a, d)] > 0.5]
    print(f"  Día {day}: vienen {len(actors_present)} actores -> {actors_present}")

PLAN (por días):
  Día 1: tomas 2, 13, 20, 27, 28, 30
  Día 2: tomas 3, 4, 8, 15, 21, 29
  Día 3: tomas 1, 6, 9, 10, 12, 26
  Día 4: tomas 14, 17, 18, 19, 23, 24
  Día 5: tomas 5, 7, 11, 16, 22, 25

COSTE TOTAL (actor-días pagados): 27

Detalle (actores que vienen por día):
  Día 1: vienen 4 actores -> [1, 3, 4, 5]
  Día 2: vienen 6 actores -> [1, 2, 5, 6, 7, 8]
  Día 3: vienen 7 actores -> [1, 2, 3, 4, 5, 6, 9]
  Día 4: vienen 3 actores -> [1, 3, 6]
  Día 5: vienen 7 actores -> [1, 2, 3, 4, 5, 8, 10]


(*)Calcula la complejidad del algoritmo

Respuesta:
Este algoritmo se divide en dos partes:
1. La construcción del modelo = O(A·T+D)
- Siendo $T= 30, A=10 y D = 30$
2. Resolución = $O(2^nbin)$
- Siendo nbin es el número de variables binarias, estas son: $T·D + A·D + D$. por ende $nbin = 900 + 300 + 30$

Diseña un algoritmo para resolver el problema por fuerza bruta

In [ ]:
import time

MAX_TOMAS_DIA = 6

# ─────────────────────────────────────────────
# FUNCIÓN DE COSTO
# Dado un día (lista de tomas), calcula cuántos
# actores únicos deben presentarse ese día
# ─────────────────────────────────────────────

def costo_dia(tomas_del_dia):
    """
    Suma los actores únicos que participan en un día.
    Un actor cobra 1 tarifa aunque grabe varias tomas ese día.
    """
    actores_presentes = set()
    for toma in tomas_del_dia:
        for actor_idx, participa in enumerate(R[toma]):
            if participa == 1:
                actores_presentes.add(actor_idx)
    return len(actores_presentes)


def costo_total(solucion):
    """
    Calcula el costo total de una solución completa.
    solucion: lista de días, cada día es una lista de índices de tomas.
    """
    return sum(costo_dia(dia) for dia in solucion)


# ─────────────────────────────────────────────
# CONVERTIR PERMUTACIÓN EN SOLUCIÓN
# Dada una permutación de tomas, las agrupa
# en bloques de MAX_TOMAS_DIA
# ─────────────────────────────────────────────

def permutacion_a_solucion(permutacion):
    """
    [0,5,2,1,...] → [[0,5,2,1,4,3], [6,7,...], ...]
    Agrupa las tomas en días de 6 en el orden dado.
    """
    solucion = []
    for i in range(0, len(permutacion), MAX_TOMAS_DIA):
        dia = list(permutacion[i:i + MAX_TOMAS_DIA])
        solucion.append(dia)
    return solucion


# ─────────────────────────────────────────────
# ALGORITMO DE FUERZA BRUTA
# Prueba todas las permutaciones posibles.
# Solo viable para N pequeño (demo con n=8)
# ─────────────────────────────────────────────

def fuerza_bruta(tomas):
    """
    Explora todas las permutaciones de las tomas dadas.
    Devuelve la mejor solución encontrada y su costo.
    """
    mejor_costo    = float('inf')  #Float inicial en espacio infinito
    mejor_solucion = None

    # itertools.permutations genera todas las permutaciones
    for permutacion in itertools.permutations(tomas):
        solucion = permutacion_a_solucion(permutacion)
        costo    = costo_total(solucion)

        if costo < mejor_costo:
            mejor_costo = costo
            mejor_solucion = solucion

    return mejor_solucion, mejor_costo


# ─────────────────────────────────────────────
# Usamos solo 8 tomas para que sea ejecutable. Con las 30 tomas, el algoritmo
# es el mismo pero computacionalmente inviable.
# ─────────────────────────────────────────────

print("=" * 50)
print("FUERZA BRUTA - Sesiones de Doblaje")
print("=" * 50)

# --- Demo con subconjunto de 8 tomas ---
tomas_demo = list(range(2,10))  # tomas 0..range-1(índice 0-based)
print(f"\nEjecutando con {len(tomas_demo)} tomas (demo viable)...")
print(f"   Permutaciones a evaluar: {len(tomas_demo)}! = {math.factorial(len(tomas_demo)):,}")

solucion, costo = fuerza_bruta(tomas_demo)

print(f"\nMejor solución encontrada:")
for i, dia in enumerate(solucion):
  # Mostramos tomas en numeración 1-based
  tomas_mostrar = [t + 1 for t in dia]
  print(f"   Día {i+1}: Tomas {tomas_mostrar}  Actores presentes: {costo_dia(dia)}")

print(f"\n Costo total: {costo} actores-día")


FUERZA BRUTA - Sesiones de Doblaje

Ejecutando con 8 tomas (demo viable)...
   Permutaciones a evaluar: 8! = 40,320

Mejor solución encontrada:
   Día 1: Tomas [3, 4, 5, 6, 7, 9]  Actores presentes: 6
   Día 2: Tomas [8, 10]  Actores presentes: 4

 Costo total: 10 actores-día


In [ ]:
 # --- Análisis de complejidad ---
print("\n" + "=" * 50)
print("COMPLEJIDAD")
print("=" * 50)
print(f"  N tomas = {T}")
print(f"  Complejidad: O(N!) en permutaciones")


=================== COMPLEJIDAD ==================
  N tomas = 30
  Complejidad: O(N!) en permutaciones
